# 03 · Segmentation and ground truth

    **Audience:** first-year biology students  
    **Time:** about 65 minutes  
    **Prerequisites:** Tutorials 01–02  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- distinguish image, semantic label and individual object
- calculate Dice and intersection-over-union
- observe why thresholding fails with uneven illumination

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · A mask is an answer to a biological question
*Semantic segmentation* labels every foreground pixel as a structure of interest. *Instance segmentation* also distinguishes each individual object. To evaluate an algorithm, we need a reference mask annotated by a person or generated from known synthetic geometry. Here we know the true mask because we made the objects ourselves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(21)
y, x = np.mgrid[:96, :96]
objects = [(25, 25, 10, 12), (62, 66, 13, 10), (65, 28, 8, 10)]
instance_labels = np.zeros((96, 96), dtype=np.uint8)
for k, (cy, cx, ry, rx) in enumerate(objects, start=1):
    inside = ((y-cy)/ry)**2 + ((x-cx)/rx)**2 <= 1
    instance_labels[inside] = k
reference_mask = instance_labels > 0
image = np.clip(0.12 + 0.16*x/96 + 0.58*reference_mask + rng.normal(0,0.12,(96,96)), 0,1)
fig, axes = plt.subplots(1,3,figsize=(10,3))
for ax, arr, title, cmap in zip(axes,[image,reference_mask,instance_labels],['Noisy image','Binary reference','Instance labels'],['gray','gray','tab10']):
    ax.imshow(arr,cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 2 · An algorithm can disagree with a reference
Create a predicted mask using an intensity threshold. False positives are background pixels predicted as foreground; false negatives are real structures that were missed.

In [ ]:
predicted_mask = image > 0.55
tp = np.logical_and(predicted_mask,reference_mask).sum()
fp = np.logical_and(predicted_mask,~reference_mask).sum()
fn = np.logical_and(~predicted_mask,reference_mask).sum()
print('True positive:',tp,'False positive:',fp,'False negative:',fn)

## 3 · Dice and IoU
**Dice** = `2 × intersection / (predicted area + reference area)`.

**IoU** = `intersection / union`. Both range from 0 (no overlap) to 1 (perfect overlap). For two empty masks we define overlap as 1 here; in research, report your empty-mask convention explicitly.

In [ ]:
def dice_score(pred, truth):
    pred, truth = np.asarray(pred,dtype=bool), np.asarray(truth,dtype=bool)
    total = pred.sum()+truth.sum()
    return 1.0 if total == 0 else float(2*np.logical_and(pred,truth).sum()/total)

def iou_score(pred, truth):
    pred, truth = np.asarray(pred,dtype=bool), np.asarray(truth,dtype=bool)
    union = np.logical_or(pred,truth).sum()
    return 1.0 if union == 0 else float(np.logical_and(pred,truth).sum()/union)

print('Dice:',round(dice_score(predicted_mask,reference_mask),3))
print('IoU:',round(iou_score(predicted_mask,reference_mask),3))
assert dice_score(reference_mask,reference_mask) == 1
assert iou_score(np.zeros((2,2),bool),np.zeros((2,2),bool)) == 1

## 4 · Visualize errors instead of trusting one score
Colors indicate agreement/disagreement: red = false positive, blue = false negative, green = correctly segmented foreground.

In [ ]:
errors = np.zeros((*image.shape,3),dtype=float)
errors[np.logical_and(predicted_mask,reference_mask)] = [0.1,0.7,0.2]
errors[np.logical_and(predicted_mask,~reference_mask)] = [1,0.1,0.1]
errors[np.logical_and(~predicted_mask,reference_mask)] = [0.2,0.3,1]
plt.figure(figsize=(5,5));plt.imshow(errors);plt.title('Green=TP, red=FP, blue=FN');plt.axis('off');plt.show()

## 5 · Choose a threshold by a held-out reference
We can sweep thresholds on **this practice image** to learn the effect. In real research, select thresholds on training/validation data and only report final performance on an untouched test set.

In [ ]:
thresholds = np.linspace(0.25,0.85,31)
scores = [dice_score(image>t,reference_mask) for t in thresholds]
best_t = float(thresholds[np.argmax(scores)])
plt.plot(thresholds,scores); plt.xlabel('Threshold');plt.ylabel('Dice (practice image)');plt.grid(alpha=.3);plt.show()
print('Best threshold on this practice image:',round(best_t,2))

## Your turn
1. Compare Dice at thresholds 0.40, 0.55 and 0.70.
2. If a background pixel is incorrectly marked foreground, is it an FP or FN?
3. Explain why a test set must not be used for choosing a threshold.

**Self-check:** For nonempty masks, Dice = `2 × IoU / (1 + IoU)` (up to rounding).

In [ ]:
iou = iou_score(predicted_mask,reference_mask)
assert np.isclose(dice_score(predicted_mask,reference_mask),2*iou/(1+iou))
print({t:round(dice_score(image>t,reference_mask),3) for t in [0.40,0.55,0.70]})

## Reflect
A high Dice score may still conceal errors on small structures or rare cases. Always inspect representative failures. Next: instead of hand-picking a threshold, let a model learn its parameters from labeled examples.